## Stage 0: Nền tảng

Cài `field.py`, `poly.py`, `mle.py`:

* Phép toán trên $\mathbb F_p$.
* Đa thức một biến cho bởi giá trị tại $0..d$, đánh giá tại $r$ bất kỳ bằng nội suy Lagrange.
* Đánh giá MLE từ bảng $2^v$ giá trị trong $O(2^v)$.

## Stage 1: Protocol sum-check tổng quát

Cài `Prover`, `Verifier` và hàm `run`:

* Prover nhận $g$ dạng callable cùng danh sách bậc theo từng biến.
* Verifier nhận claim ban đầu, danh sách bậc, và một oracle để đánh giá $g$ ở kiểm tra cuối.
* `run(prover, verifier, rs=None)` trả về accept/reject và transcript; `rs` cho phép ép trước các challenge.

In [15]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

from share.interactive import Party, Simulation
from share.sumcheck import Verifier, Prover
from share.field import F
from share.poly import UniPoly
from share.mle import MLE

from itertools import product
from random import randint, seed
from types import MethodType

## Stage 2
## Stage 2: Tấn công và soundness

**(a)** Prover gian lận biết trước $r_1$. Hãy claim $H' = H + 1$ mà V vẫn chấp nhận.

**(b)** Với $p = 97$, prover gian lận **không** biết trước $r_1$. Nó chơi chiến lược tốt nhất có thể ở vòng 1, rồi trung thực từ vòng 2. Chạy nhiều lần, đo tỉ lệ V chấp nhận.


In [2]:
def g(x, y, z):
    return F(2*x*x*x + x*z + y*z)

prover = Prover("Alice", g, [3, 1, 1])
verifier = Verifier("Bob", g, [3, 1, 1], prover.sum())
prover.set_dst(verifier)

seed(1337)
sim = Simulation([prover, verifier])
sim.run()
print(sim.result)
sim.print_log()

{'Bob': True}
Alice -> Bob: ('g1', [F(1), F(11), F(69), F(223)])
Bob -> Alice: ('r1', F(2134139354768280835))
Alice -> Bob: ('g2', [F(2227249431095291974), F(2227249431095291975)])
Bob -> Alice: ('r2', F(1636871794675947429))
Alice -> Bob: ('g3', [F(1199476542770352545), F(358801673787192907)])


In [3]:
fake_claim = prover.sum() + 1
know_r1 = F(2134139354768280835)

def prover_run(self):
    assert self.dst is not None, "Please set verifier"
    dst = self.dst
    g = self.g
    degs = self.degs
    v = len(degs)
    rs = []

    s1_r1 = self._sum(lambda *x: g(know_r1, *x) , v-1)
    alpha = (fake_claim - 2*s1_r1) / (1 - 2*know_r1)
    beta  = (s1_r1 - fake_claim*know_r1) / (1 - 2*know_r1)
    assert alpha + 2*beta == fake_claim
    assert know_r1*alpha + beta == s1_r1

    ev = (beta, alpha + beta)

    yield (dst, (f"g1", ev))
    _, (_, r) = yield
    rs.append(r)

    for i in range(1, v):
        ev = [self._sum(lambda *x: g(*rs, j, *x) , v-i-1) for j in range(degs[i]+1)]
        yield (dst, (f"g{i+1}", ev))
        _, (_, r) = yield
        rs.append(r)

prover.run = MethodType(prover_run, prover)
verifier.claims = [fake_claim]

seed(1337)
sim = Simulation([prover, verifier])
sim.run()
print(sim.result)
sim.print_log()

{'Bob': True}
Alice -> Bob: ('g1', (F(1284338016539427677), F(1021504992674266287)))
Bob -> Alice: ('r1', F(2134139354768280835))
Alice -> Bob: ('g2', [F(2227249431095291974), F(2227249431095291975)])
Bob -> Alice: ('r2', F(1636871794675947429))
Alice -> Bob: ('g3', [F(1199476542770352545), F(358801673787192907)])


In [4]:
fake_claim = prover.sum() + 1

def prover_run(self):
    assert self.dst is not None, "Please set verifier"
    dst = self.dst
    g = self.g
    degs = self.degs
    v = len(degs)
    rs = []

    ev = (1, fake_claim-1)
    yield (dst, (f"g1", ev))
    _, (_, r) = yield
    rs.append(r)

    for i in range(1, v):
        ev = [self._sum(lambda *x: g(*rs, j, *x) , v-i-1) for j in range(degs[i]+1)]
        yield (dst, (f"g{i+1}", ev))
        _, (_, r) = yield
        rs.append(r)

prover.run = MethodType(prover_run, prover)
verifier.claims = [fake_claim]
F.p = 97

count = 0
for _ in range(10000):
    sim = Simulation([prover, verifier])
    sim.run()
    if sim.result['Bob']:
        count += 1

print(f"{count/100}%")

2.97%


## Stage 3: #SAT bằng arithmetization

* Biểu diễn công thức Boolean dạng cây.
* Viết hàm arithmetize trả về callable $g$ và bậc theo từng biến.
* Chạy sum-check để đếm số nghiệm thỏa.


In [ ]:
AND = "and"
OR  = "or"
NOT = "not"
VAR = "var"

bf = \
(AND, 
    (OR,
        (VAR, 1),
        (NOT, (VAR, 2))
    ),
    (AND,
        (OR,
            (VAR, 2),
            (VAR, 3)
        ),
        (OR,
            (NOT, (VAR, 1)),
            (NOT, (VAR, 4))
        )
    )
)

def arith(bf):
    op = bf[0]
    if op == VAR:
        return lambda x: x, [(bf[1], 1)]
    if op == NOT:
        f, degs = arith(bf[1])
        return lambda *x: 1 - f(*x), degs
    if op == AND or op == OR:
        fl, degsl = arith(bf[1])
        fr, degsr = arith(bf[2])

        ml = [0] * len(degsl)
        mr = [0] * len(degsr)

        pl = 0
        pr = 0
        p  = 0

        degs = []
        
        while degsl or degsr:
            l0 = degsl[0][0] if degsl else float("inf")
            r0 = degsr[0][0] if degsr else float("inf")
            if l0 == r0:
                ml[pl] = p
                mr[pr] = p
                pl += 1
                pr += 1
                p  += 1
                degs.append((l0, degsl[0][1] + degsr[0][1]))
                degsr.pop(0)
                degsl.pop(0)
            elif l0 < r0:
                ml[pl] = p
                pl += 1
                p  += 1
                degs.append((l0, degsl[0][1]))
                degsl.pop(0)
            else:
                mr[pr] = p
                pr += 1
                p  += 1
                degs.append((r0, degsr[0][1]))
                degsr.pop(0)

        flf = lambda x: fl(*[x[i] for i in ml])
        frf = lambda x: fr(*[x[i] for i in mr])

        if op == AND:
            f = lambda *x: flf(x) * frf(x)
        else:
            f = lambda *x: flf(x) + frf(x) - flf(x) * frf(x)


    return f, degs            

g, degs = arith(bf)
print(degs)

# Another idea?

print(g(2, 3, 4, 5))

[(1, 2), (2, 2), (3, 1), (4, 1)]
180


In [14]:
degs = [deg[1] for deg in degs]
prover = Prover("#SAT Prover", g, degs)
print(prover.sum())
verifier = Verifier("#SAT Verifier", g, degs, prover.sum())
prover.set_dst(verifier)
sim = Simulation([prover, verifier])
sim.run()
print(sim.result)

5
{'#SAT Verifier': True}


## Stage 4: So sánh với MLE của φ

* Tính $\tilde\varphi$ bằng bảng chân lý và `mle_eval`, rồi so với $g$ của Stage 3.
* Kiểm chứng $\#\mathrm{SAT}(\varphi) = 2^n \cdot \tilde\varphi(2^{-1},\dots,2^{-1})$, rồi rút ra vì sao verifier không thể dùng $\tilde\varphi$.


In [ ]:
phi = MLE([g(*x) for x in product([0, 1], repeat=len(degs))])
print(phi(2, 3, 4, 5))
half = 1/F(2)
print(2**4*phi(half, half, half, half))

F(48)
F(5)
